# S0 official-harness gate (preflight only)

**This notebook never submits anything.** Do not press *Submit*. Run it interactively as a NEW notebook (a Copy & Edit of the official starter, plus the private dataset that holds `GEMMA_S0_PREFLIGHT_PACK.zip`). Never save it as a new version of the notebook that is already running.

Cells marked **ours** are code from the pack. Cells marked **STARTER** are the official starter's own cells, which this notebook deliberately does not reproduce because they were not available when it was written.

Status words used throughout: PASS, WARN, FAIL, BLOCKED. A BLOCKED check is not a pass.

In [ ]:
# CELL 1 (ours) - safety rails. Nothing here touches the competition or the already-running notebook.
import os, sys, json, glob, shutil, subprocess, platform, zipfile, sysconfig, importlib, importlib.metadata as md
from pathlib import Path

ROOT = Path(os.environ.get("S0_PREFLIGHT_ROOT", "/kaggle"))      # overridden only by the local rehearsal
K_IN, K_WORK = ROOT / "input", ROOT / "working"
OUT = K_WORK / "preflight_out"
OUT.mkdir(parents=True, exist_ok=True)

EXPECTED_S0_SHA256 = "359a50986e9c23926cd99da853ad30fa399291947a9e9fc68b5906339156cf1b"
FORBIDDEN_OUTPUTS = [K_WORK / "submission.zip", K_WORK / "submission.parquet"]
for p in FORBIDDEN_OUTPUTS:
    assert not p.exists(), f"{p} already exists. This notebook must never produce a competition submission file. Stop."

print("python", platform.python_version(), "| working dir", K_WORK)
print("outputs go to", OUT)
print("REMINDER: never press Submit here; never save this as a new version of the already-running notebook.")

In [ ]:
# CELL 2 (ours) - locate the preflight pack, import the helpers, self-test them
hits = glob.glob(str(K_IN / "**" / "preflight_tools.py"), recursive=True)
if hits:
    PACK_DIR = Path(hits[0]).parent.parent
else:
    zips = glob.glob(str(K_IN / "**" / "GEMMA_S0_PREFLIGHT_PACK.zip"), recursive=True)
    assert zips, "Attach the private dataset that holds GEMMA_S0_PREFLIGHT_PACK.zip (or its extracted files)."
    PACK_DIR = K_WORK / "preflight" / "pack"
    PACK_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zips[0]) as z:
        z.extractall(PACK_DIR)
print("pack directory:", PACK_DIR)
sys.path.insert(0, str(PACK_DIR / "scripts"))
import preflight_tools as pt
st = subprocess.run([sys.executable, str(PACK_DIR / "scripts" / "preflight_tools.py"), "selftest"], capture_output=True, text=True)
print(st.stdout[-700:])
assert st.returncode == 0, "helper self-test failed: do not trust the cells that follow"

In [ ]:
# CELL 3 (ours) - what is mounted, and which Python and packages are present (read-only)
inv = pt.inventory(str(K_IN))
print(json.dumps(inv, indent=2))
(OUT / "inputs_inventory.json").write_text(json.dumps(inv, indent=2))
for key in ("harness_readme", "sample_submission", "wheels", "tasks_jsonl", "snapshots"):
    print(f"{key:18s}", inv.get(key, "NOT FOUND"))
print(subprocess.run("python3 --version; pip list 2>/dev/null | grep -i -E '^(swegemma|adk-submission|adk-eval-core|google-adk|litellm|vllm|pyyaml) '; nvidia-smi -L 2>&1 | head -5",
                     shell=True, capture_output=True, text=True).stdout)

## Cell 4 (STARTER) - run the starter's own setup cells now, unchanged

Run the starter cells that install the competition wheels offline and import the harness. Stop before the starter cell that launches the model server; that one belongs to Cell 11. Do not run the starter's final cell that zips an agent or writes a submission file.

Why this is a manual step: the install command, wheel paths and server settings are the starter's, and they must be the ones the platform uses. If an install cell needs the network, stop: this gate is offline-only.

Accelerator tip: Cells 1 to 10 need no GPU. You can run them with the accelerator off, then switch on the starter's GPU setting for Cell 11 (the session restarts when you switch, so rerun Cells 1, 2, 3, 4 and 5 afterwards).

In [ ]:
# CELL 5 (ours) - AFTER the starter's install/import cells: versions and locations of the official packages
report = {}
for mod, dist in [("swegemma", "swegemma"), ("adk_submission", "adk-submission"),
                  ("adk_eval_core", "adk-eval-core"), ("google.adk", "google-adk")]:
    try:
        m = importlib.import_module(mod)
        try:
            ver = md.version(dist)
        except Exception:
            ver = getattr(m, "__version__", "unknown")
        report[mod] = {"version": ver, "file": getattr(m, "__file__", None)}
    except Exception as e:
        report[mod] = {"error": f"{type(e).__name__}: {e}"}
print(json.dumps(report, indent=2))
(OUT / "package_versions.json").write_text(json.dumps(report, indent=2))
missing = [k for k, v in report.items() if "error" in v]
assert not missing, f"official packages not importable: {missing}. Run the starter's offline install cells first."

In [ ]:
# CELL 6 (ours) - verify the candidate hash, extract it, run the static audit
S0_ZIP = PACK_DIR / "candidate" / "submission.zip"
sha = pt.sha256_file(S0_ZIP)
print("S0 sha256:", sha)
assert sha == EXPECTED_S0_SHA256, "candidate hash differs from the validated S0 hash: STOP"
S0_DIR = K_WORK / "preflight" / "s0_agent"
if S0_DIR.exists():
    shutil.rmtree(S0_DIR)
print(pt.safe_extract(S0_ZIP, S0_DIR))
audit = subprocess.run([sys.executable, str(PACK_DIR / "scripts" / "validate_s0_v2.py"), str(S0_ZIP),
                        "--json-out", str(OUT / "static_audit.json")], capture_output=True, text=True)
print(audit.stdout[-3500:])
print(audit.stderr[-800:])
assert audit.returncode == 0, "static audit reported a FAIL"

In [ ]:
# CELL 7 (ours) - compare S0 with the OFFICIAL sample_submission (settles the eval_config layout question)
sample_dirs = inv.get("sample_submission", [])
assert sample_dirs, "official sample_submission not found: layout comparison is BLOCKED"
cmp_res = pt.compare_with_sample(sample_dirs[0], S0_DIR)
print("\n".join(cmp_res["lines"]))
print("VERDICT:", cmp_res["verdict"])
(OUT / "sample_comparison.json").write_text(json.dumps(cmp_res, indent=2))
assert cmp_res["verdict"] != "MISMATCH", "official sample does not use the layout S0 uses: STOP and re-check the contract"

In [ ]:
# CELL 8 (ours) - read the OFFICIAL eval_config loader source without executing it (AST only)
cands = list(inv.get("inference_py", [])) + list(inv.get("metric_py", []))
site = Path(sysconfig.get_paths()["purelib"])
for d in site.glob("swegemma*"):
    cands += [str(x) for x in d.rglob("inference.py")]
cands = list(dict.fromkeys(cands))
print("candidate official files:", cands)
rep = None
for c in cands:
    if c.endswith("inference.py"):
        rep = pt.official_source_report(c, ["load_submission_eval_config", "setup_vllm_server", "validate_single_declared_model"])
        break
if rep is None:
    print("BLOCKED: scripts/inference.py was not found in the mounted inputs or in site-packages")
else:
    for f, s in rep["functions"].items():
        print("=====", f, "=====")
        print(s or "(not found)")
    print({k: v for k, v in rep.items() if k != "functions"})
    (OUT / "official_loader_source.txt").write_text(
        "\n\n".join(f"### {f}\n{s}" for f, s in rep["functions"].items() if s) or "no function found")

In [ ]:
# CELL 9 (ours) - official validators, found by introspection (never guessed). Exit code 3 means BLOCKED.
res = subprocess.run([sys.executable, str(PACK_DIR / "scripts" / "official_harness_check.py"), str(S0_ZIP)],
                     capture_output=True, text=True)
print(res.stdout[-5000:])
print(res.stderr[-1500:])
print("exit:", res.returncode)
(OUT / "official_validators.txt").write_text(res.stdout + "\n--- stderr ---\n" + res.stderr + f"\nexit={res.returncode}\n")

In [ ]:
# CELL 10 (ours) - is there a DOCUMENTED fake/scripted-model option in the official CLI? (look, do not invent)
h = subprocess.run("swegemma eval --help", shell=True, capture_output=True, text=True)
(OUT / "swegemma_eval_help.txt").write_text(h.stdout + h.stderr)
print(h.stdout[:3000])
print(h.stderr[:500])
scan = pt.help_scan(h.stdout + h.stderr)
print(json.dumps(scan, indent=2))
print("FAKE-MODEL REPLAY:", "an option may exist, read the hits above" if scan["documented_fake_model_option_found"]
      else "BLOCKED: no documented fake/scripted-model option found in the CLI help")

## Cell 11 (STARTER, edit don't rewrite) - ONE fixed task with the real model

These are edit instructions, not code, because the starter's cells were not available to the author. They are based on a third-party description of the starter, so check each against your starter.

Use the starter's own cells that (a) launch the local model server, (b) build the evaluator and (c) run it. Make exactly these edits and no others:

1. Point the agent-directory variable at `S0_DIR` (the folder created in Cell 6). Look for `AGENT_DIR` or the path of `sample_submission`.
2. Restrict the task list to the single id in `TASK_ID` (Cell 12 prints it). Look for the line that loads or slices the tasks.
3. Send results to a new, empty folder such as `/kaggle/working/results_s0_smoke`.
4. Keep the starter's budget prelude exactly as it is; it reads `eval_config.yaml` from the agent directory. Do NOT add explicit tool-call, time or turn overrides, because that would hide whether S0's own `eval_config.yaml` is applied.
5. Do NOT run the starter's final cell that zips an agent or writes a submission file.

Run interactively. If the evaluator is still running 15 minutes after the model server reports ready, press Interrupt, then run Cell 14 to collect what exists. Never use *Save & Run All* or commit this notebook while the smoke test is incomplete. S0's own limits (5 min, 50 calls, 100 turns, 300 s per command) bound the agent; server start-up is not part of them.

In [ ]:
# CELL 12 (ours) - pick ONE fixed public task id deterministically (only the id is read; no patches or answers)
tasks_path = (inv.get("tasks_jsonl") or [None])[0]
assert tasks_path, "tasks.jsonl not found"
ids = []
with open(tasks_path, encoding="utf-8") as fh:
    for line in fh:
        d = json.loads(line)
        ids.append(d.get("instance_id") or d.get("id"))
ids = [i for i in ids if i]
TASK_ID = sorted(ids)[0]
print("fixed task id:", TASK_ID, f"(first of {len(ids)} public task ids in sorted order)")
(OUT / "smoke_task_id.txt").write_text(TASK_ID)

## Cell 13 - the smoke run happens here (Cell 11 edits applied to the starter's cells)

Nothing to paste in this cell. When the run has finished, or after you interrupted it, continue with Cell 14.

In [ ]:
# CELL 14 (ours) - AFTER the smoke run: collect artifacts, check the budget the agent actually saw, audit tool calls
cands = {}
for pat in ("task_results.jsonl", "summary.json"):
    for p in glob.glob(str(K_WORK / "**" / pat), recursive=True):
        if "preflight_out" in p:
            continue
        parent = str(Path(p).parent)
        cands[parent] = cands.get(parent, 0) + 1
print("results directories found:", cands)
assert cands, "no results directory found: did the smoke run finish? (interrupt it, then rerun this cell to collect partial output)"
RES = Path(sorted(cands, key=lambda k: -cands[k])[0])
dest = OUT / "smoke"
dest.mkdir(exist_ok=True)
for name in ("summary.json", "task_results.jsonl"):
    if (RES / name).exists():
        shutil.copy2(RES / name, dest / name)
for sub in ("patches", "test_outputs", "traces", "logs"):
    if (RES / sub).exists():
        shutil.copytree(RES / sub, dest / sub, dirs_exist_ok=True)
print("copied from", RES)

if (dest / "task_results.jsonl").exists():          # scalar fields only; patch bodies are not printed
    for line in (dest / "task_results.jsonl").read_text().splitlines():
        rec = json.loads(line)
        print({k: (v if isinstance(v, (int, float, bool)) or v is None
                   else ("<omitted>" if any(t in k.lower() for t in ("patch", "diff")) else str(v)[:200]))
               for k, v in rec.items()})

traces = sorted((dest / "traces").glob("trace_*.json")) if (dest / "traces").exists() else []
if traces:
    r = pt.extract_first_user_message(traces[0], OUT / "first_user_message.txt")
    print(r)
    if r["ok"]:
        chk = subprocess.run([sys.executable, str(PACK_DIR / "scripts" / "official_harness_check.py"),
                              "--check-budget-prompt", str(OUT / "first_user_message.txt")], capture_output=True, text=True)
        print(chk.stdout)
    print(json.dumps(pt.trace_tools(traces[0]), indent=2))
else:
    print("no trace JSON found: the budget and tool-call assertions are BLOCKED")

In [ ]:
# CELL 15 (ours) - manifest, hashes, archive of outputs, final guard rails. STOP after this cell.
manifest = {}
for p in sorted(OUT.rglob("*")):
    if p.is_file() and p.name != "preflight_manifest.json":
        manifest[str(p.relative_to(OUT))] = {"bytes": p.stat().st_size, "sha256": pt.sha256_file(p)}
(OUT / "preflight_manifest.json").write_text(json.dumps(manifest, indent=2))
archive = shutil.make_archive(str(K_WORK / "s0_preflight_outputs"), "zip", str(OUT))
print("outputs archive:", archive)
for p in FORBIDDEN_OUTPUTS:
    assert not p.exists(), f"{p} exists: delete it. Nothing from this notebook may be submitted."
print("STOP HERE. Download s0_preflight_outputs.zip, do NOT press Submit, and end the session.")